# Qwen3.6-27B + CompactDS Wikipedia-DPR RAG

First port of CompactDS retrieval (Lyu et al. arxiv 2507.01297) to Qwen3.6-27B dense.

## Plan (Path B — fast, 5h budget)

1. Setup + load Qwen3.6-27B (~4 min)
2. Download CompactDS Wikipedia-DPR subset (~5.2GB, 20 min)
3. Load Contriever-msmarco encoder + FAISS index (~2 min)
4. Define 3 generation methods:
   - **baseline**: greedy (no RAG)
   - **RAG**: greedy + top-5 passages prepended
   - **RAG+CoN**: RAG with Chain-of-Note instruction (2311.09210)
5. Eval 30 SuperGPQA prompts × 3 methods (~4h)
6. Upload results

## Expected

| method | SuperGPQA projected |
|---|---|
| Qwen3.6-27B greedy (baseline) | 0.55-0.70 (sample variance) |
| + RAG top-5 Wikipedia | +2-4pp |
| + RAG + Chain-of-Note | +1-2pp adicional |

## Decision rule

- Lift ≥ +3pp (RAG): **VALIDATED**, scale to Path A (Qwen3-Embedding rebuild)
- Lift [0, +3pp]: marginal, try Path C (PeS2o+Arxiv)
- Lift < 0: RAG hurts (distraction), pivot

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5' in CONFIG_MAPPING_NAMES
except Exception: ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer', 'faiss-cpu')
    pip('uninstall', '-y', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    pip('install','--no-cache-dir','flash-linear-attention')
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'): del sys.modules[m]

pip('install', '-q', 'faiss-cpu')

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, re, time, random
import numpy as np
import torch
from tqdm.auto import tqdm
from collections import Counter
from safetensors import safe_open
from huggingface_hub import snapshot_download, HfApi, create_repo
import faiss
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/rag_eval'); OUT.mkdir(exist_ok=True)
MODEL_ID = 'Qwen/Qwen3.6-27B'
HF_STAGE_B = 'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b'
HF_TARGET = 'caiovicentino1/qwen36-27b-compactds-rag'
COMPACTDS_REPO = 'alrope/CompactDS-102GB'

# Hyperparams
TOP_K = 5
MAX_PASSAGE_TOKENS = 200  # truncate each passage
MAX_NEW = 3500
TEMP_GREEDY = 0.0
SEED = 42
N_EVAL = 30
FORCE_SUFFIX = '\n</think>\n\nFinal answer: \\boxed{'

print(f'Setup done: Qwen3.6-27B + CompactDS Wikipedia-DPR RAG, N={N_EVAL}, top_k={TOP_K}')

## Cell 2 — Load Qwen3.6-27B

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print(f'Qwen3.6-27B loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 3 — Download CompactDS Wikipedia-DPR subset + inspect

In [ ]:
compact_dir = snapshot_download(
    COMPACTDS_REPO, repo_type='dataset', cache_dir='/content/cache',
    allow_patterns=['wikipedia_dpr/*', '*.json', 'README*'])

print(f'Downloaded to: {compact_dir}')

# Inspect structure
wiki_dir = Path(compact_dir) / 'wikipedia_dpr'
if wiki_dir.exists():
    for f in sorted(wiki_dir.iterdir())[:20]:
        size_mb = f.stat().st_size / 1e6
        print(f'  {f.name}: {size_mb:.1f} MB')
else:
    print('wikipedia_dpr/ not found, listing root:')
    for f in sorted(Path(compact_dir).iterdir())[:20]:
        print(f'  {f.name}')

## Cell 4 — Load Contriever + FAISS index

In [ ]:
from transformers import AutoTokenizer as ContrieverTok, AutoModel

# Load Contriever encoder
ctk = ContrieverTok.from_pretrained('facebook/contriever-msmarco')
cenc = AutoModel.from_pretrained('facebook/contriever-msmarco').cuda()
cenc.eval()
print(f'Contriever loaded: {sum(p.numel() for p in cenc.parameters())/1e6:.0f}M params')

def mean_pooling(token_embeddings, mask):
    token_embeddings = token_embeddings.masked_fill(~mask[..., None].bool(), 0.)
    sentence_embeddings = token_embeddings.sum(dim=1) / mask.sum(dim=1)[..., None]
    return sentence_embeddings

def encode_query(text):
    inputs = ctk(text, padding=True, truncation=True, max_length=256, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = cenc(**inputs)
        emb = mean_pooling(out[0], inputs['attention_mask'])
    return emb.cpu().numpy().astype('float32')

# Load FAISS index (mmap'd to save RAM)
# Find index file
index_files = list(wiki_dir.rglob('*.index')) + list(wiki_dir.rglob('*.faiss'))
print(f'Index files found: {[f.name for f in index_files]}')

if index_files:
    index_path = str(index_files[0])
    print(f'Loading FAISS index from {index_path}')
    index = faiss.read_index(index_path, faiss.IO_FLAG_MMAP | faiss.IO_FLAG_READ_ONLY)
    print(f'  index.ntotal = {index.ntotal:,} vectors')
    print(f'  index.d = {index.d}')
else:
    raise RuntimeError('FAISS index not found in wikipedia_dpr/')

# Load passages (JSONL or similar)
passage_files = (list(wiki_dir.rglob('*.jsonl')) + list(wiki_dir.rglob('*.json')) +
                  list(wiki_dir.rglob('passages*')))
passage_files = [p for p in passage_files if p.stat().st_size > 1e6]  # skip tiny json
print(f'Passage files: {[(f.name, f.stat().st_size/1e9) for f in passage_files]}')

## Cell 5 — Build passage loader + retrieve helper

In [ ]:
# Load passages lazily — depending on format (jsonl vs parquet vs mmap)
# Try jsonl first
passages_by_id = None
if passage_files:
    pf = passage_files[0]
    print(f'Loading passages from {pf.name}...')
    if pf.suffix == '.jsonl':
        passages_by_id = {}
        with open(pf) as f:
            for i, line in enumerate(tqdm(f, desc='load passages')):
                try:
                    obj = json.loads(line)
                    passages_by_id[i] = obj.get('text', obj.get('contents', obj.get('passage', '')))
                except: continue
                if i >= 5_000_000: break  # cap at 5M for memory
        print(f'Loaded {len(passages_by_id):,} passages')
    elif pf.suffix == '.parquet':
        import pyarrow.parquet as pq
        tb = pq.read_table(pf, columns=['text']) if 'text' in pq.read_schema(pf).names else pq.read_table(pf)
        passages_by_id = {i: r.as_py() for i, r in enumerate(tb['text'])}
        print(f'Loaded {len(passages_by_id):,} passages from parquet')

# If not found, print available files for debug
if passages_by_id is None or len(passages_by_id) == 0:
    print('No passages loaded — inspect file format:')
    for f in list(wiki_dir.rglob('*'))[:30]:
        print(f'  {f.relative_to(wiki_dir)}: {f.stat().st_size/1e6:.1f} MB')
    raise RuntimeError('No passages available')

def retrieve(query_text, top_k=TOP_K):
    q_emb = encode_query(query_text)
    faiss.normalize_L2(q_emb)
    D, I = index.search(q_emb, top_k)
    passages = []
    for idx in I[0]:
        if idx < 0: continue
        p = passages_by_id.get(int(idx))
        if p:
            # Truncate to MAX_PASSAGE_TOKENS
            words = p.split()[:MAX_PASSAGE_TOKENS]
            passages.append(' '.join(words))
    return passages

# Quick test
test = retrieve('What is the capital of France?', top_k=3)
for i, p in enumerate(test):
    print(f'[{i}] {p[:150]}...')

## Cell 6 — Generation helpers: greedy, RAG, RAG+CoN

In [ ]:
def extract_answer(text):
    post = text.split('</think>')[-1] if '</think>' in text else text
    for pattern in [r'\\boxed\{([A-J])\}',
                    r'[Aa]nswer\s*(?:is\s*)?[:=]?\s*\*?\*?\(?([A-J])\)?\*?\*?',
                    r'^\s*\(?([A-J])\)?\s*\.?\s*$']:
        m = re.search(pattern, post, re.MULTILINE)
        if m: return m.group(1).upper()
    m = re.findall(r'\\boxed\{([A-J])\}', text)
    if m: return m[-1]
    tail = post[-300:] if post else text[-300:]
    letters = re.findall(r'\b([A-J])\b', tail)
    return letters[-1] if letters else None

def force_close(full_ids, prompt_len):
    gen = tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=False)
    if '</think>' in gen:
        return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True)
    full = tok.decode(full_ids.tolist(), skip_special_tokens=False) + FORCE_SUFFIX
    ctx = tok(full, return_tensors='pt', add_special_tokens=False).input_ids.cuda()
    with torch.no_grad():
        out = model.generate(ctx, max_new_tokens=15, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    suf = tok.decode(out[0, ctx.shape[1]:].tolist(), skip_special_tokens=True)
    return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True) + FORCE_SUFFIX + suf

def fmt_base(q, opts):
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    content = (f'Answer the following multiple-choice question. '
               f'Reason step by step, then put the letter in \\boxed{{}}.\n\n'
               f'Question: {q}\n\nOptions:\n{choices}')
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=True)

def fmt_rag(q, opts, passages, con=False):
    # Reverse order: most relevant closer to query (CompactDS paper §3.2)
    passages_ordered = list(reversed(passages))
    ctx = '\n\n'.join(f'[passage {len(passages)-i}] {p}' for i, p in enumerate(passages_ordered))
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    system = ('You are an expert reasoner. Use the retrieved passages ONLY if they are '
              'directly relevant. Ignore noisy or off-topic passages.')
    if con:
        system += (' FIRST write brief notes about which passages are relevant. '
                   'THEN answer using only the relevant information. '
                   'If no passage is relevant, answer using your own knowledge.')
    user_content = (f'<context>\n{ctx}\n</context>\n\n'
                    f'Question: {q}\n\nOptions:\n{choices}\n\n'
                    f'Reason step by step, then put the letter in \\boxed{{}}.')
    return tok.apply_chat_template(
        [{'role':'system','content':system},
         {'role':'user','content':user_content}],
        tokenize=False, add_generation_prompt=True,
        enable_thinking=True)

def gen_greedy(prompt):
    ids = tok(prompt, return_tensors='pt').input_ids.cuda()
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    return extract_answer(force_close(out[0], ids.shape[1]))

def method_baseline(q, opts):
    return gen_greedy(fmt_base(q, opts))

def method_rag(q, opts, con=False):
    passages = retrieve(q, top_k=TOP_K)
    prompt = fmt_rag(q, opts, passages, con=con)
    return gen_greedy(prompt), passages

print('helpers ready')

## Cell 7 — Load eval set + run 3-way comparison

In [ ]:
corpus = snapshot_download(HF_STAGE_B, repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

questions = []
seen = set()
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q = meta['question']
        if q in seen: continue
        seen.add(q)
        opts = json.loads(meta['options'])
        if len(opts) != 10: continue
        questions.append(dict(question=q, options=opts, gold_letter=meta['gold_letter']))

random.seed(SEED); random.shuffle(questions)
eval_set = questions[:N_EVAL]
print(f'{len(eval_set)} eval prompts (seed={SEED})')

results = {'baseline': [], 'rag': [], 'rag_con': []}
details = []
t0 = time.time()

for qi, q in enumerate(tqdm(eval_set, desc='RAG eval')):
    gold = q['gold_letter']
    # Baseline
    try:
        b = method_baseline(q['question'], q['options'])
        results['baseline'].append((b, b == gold))
    except Exception as e:
        print(f'baseline err: {e}'); results['baseline'].append((None, False))
    # RAG
    try:
        r, passages = method_rag(q['question'], q['options'], con=False)
        results['rag'].append((r, r == gold))
    except Exception as e:
        print(f'rag err: {e}'); results['rag'].append((None, False)); passages = []
    # RAG + CoN
    try:
        rc, _ = method_rag(q['question'], q['options'], con=True)
        results['rag_con'].append((rc, rc == gold))
    except Exception as e:
        print(f'rag_con err: {e}'); results['rag_con'].append((None, False))

    details.append({'qi': qi, 'gold': gold,
                    'baseline': b, 'rag': r, 'rag_con': rc,
                    'n_passages': len(passages),
                    'first_passage': passages[0][:200] if passages else None})

    if (qi+1) % 3 == 0:
        b_acc = np.mean([r[1] for r in results['baseline']])
        r_acc = np.mean([r[1] for r in results['rag']])
        rc_acc = np.mean([r[1] for r in results['rag_con']])
        print(f'  [{qi+1}/{len(eval_set)}]  baseline={b_acc:.3f}  rag={r_acc:.3f}  rag_con={rc_acc:.3f}  Δ={(r_acc-b_acc)*100:+.1f}pp  CoN={(rc_acc-b_acc)*100:+.1f}pp')
        # Crash-safe
        partial = {k: [(r[0], bool(r[1])) for r in v] for k, v in results.items()}
        with open(OUT / 'partial.json', 'w') as f:
            json.dump({'results': partial, 'details': details}, f, indent=2)

b_acc = np.mean([r[1] for r in results['baseline']])
r_acc = np.mean([r[1] for r in results['rag']])
rc_acc = np.mean([r[1] for r in results['rag_con']])
delta_rag = (r_acc - b_acc) * 100
delta_con = (rc_acc - b_acc) * 100

print(f'\n{"="*60}')
print(f'  FINAL (n={len(eval_set)}, {(time.time()-t0)/60:.0f} min)')
print(f'{"="*60}')
print(f'  baseline:  {b_acc:.3f}  ({sum(r[1] for r in results["baseline"])}/{len(results["baseline"])})')
print(f'  + RAG:     {r_acc:.3f}  Δ={delta_rag:+.1f}pp')
print(f'  + RAG+CoN: {rc_acc:.3f}  Δ={delta_con:+.1f}pp')
print(f'{"="*60}')

if delta_rag >= 3:
    verdict = '✅ VALIDATED — RAG gives +{:.1f}pp'.format(delta_rag)
elif delta_rag >= 0:
    verdict = '🟡 MARGINAL ({:.1f}pp)'.format(delta_rag)
else:
    verdict = '❌ RAG HURTS ({:.1f}pp — likely distraction)'.format(delta_rag)
print(f'  verdict:   {verdict}')

## Cell 8 — Upload

In [ ]:
summary = {
    'model': MODEL_ID,
    'retriever': 'facebook/contriever-msmarco',
    'datastore': 'CompactDS Wikipedia-DPR subset',
    'top_k': TOP_K,
    'max_passage_tokens': MAX_PASSAGE_TOKENS,
    'n_eval': len(eval_set), 'seed': SEED,
    'results': {
        'baseline': float(b_acc),
        'rag': float(r_acc),
        'rag_con': float(rc_acc),
    },
    'deltas_pp': {
        'rag_vs_baseline': float(delta_rag),
        'rag_con_vs_baseline': float(delta_con),
        'con_vs_rag': float((rc_acc - r_acc) * 100),
    },
    'verdict': verdict.split()[-1] if '✅' in verdict else verdict.split()[0],
    'eval_date': time.strftime('%Y-%m-%d'),
}
with open(OUT / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
with open(OUT / 'details.json', 'w') as f:
    json.dump(details, f, indent=2)

api = HfApi()
create_repo(HF_TARGET, repo_type='model', private=False, exist_ok=True)
for fn in ['summary.json', 'details.json', 'partial.json']:
    fp = OUT / fn
    if fp.exists():
        api.upload_file(path_or_fileobj=str(fp), path_in_repo=fn,
                        repo_id=HF_TARGET, repo_type='model',
                        commit_message=f'RAG eval: {fn}')
        print(f'OK {fn}')
print(f'\n✅ Results: https://huggingface.co/{HF_TARGET}')